In [9]:
from pathlib import Path
import os
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import gcamreader

In [20]:
# =========================================================
# Config
# =========================================================
PROJECT_PATH   = Path("/data/project/tae/gcam-core")
DB_REL_PATH    = "../output"
DB_FILE        = "database_basexdb_korea_2035"
QUERY_FILE     = Path("..") / "output" / "queries" / "Main_queries.xml"

REGION         = "South Korea"
SCENARIOS      = ["Current-Policies-Med", "Enhanced-Ambition-Med"]

EJ_TO_MTOE     = 23.8846
Y_TITLE        = "Mtoe"
YEARS_2005_2050 = list(range(2005, 2051, 5))
YEARS_2015_2050 = list(range(2015, 2051, 5))
FIG_KW         = dict(width=800, height=650, scale=2)  # for pio.write_image

In [21]:
# =========================================================
# Color / pattern palettes
# =========================================================
TECH_COLORS = {
    "BF": "#000000", "BF-CCS": "grey", "BF-H2": "#E4EAED", "BF-Biomass": "#099B43",
    "EAF-scrap": "#492681", "DRI-EAF": "#0057B5", "DRI-EAF-CCS": "#4DDADC", "DRI-EAF-H2": "#FED30B",
}
TECH_PATTERNS = {"BF-CCS": "/", "DRI-EAF-CCS": "/", "DRI-EAF-H2": "\\"}

FUEL_COLORS = {
    "hydrogen": "#FED30B", "electricity": "#0057B5",
    "biomass CCS": "rgb(153, 201, 69)", "biomass": "#099B43",
    "gas CCS": "#33A8E2", "gas": "#0186E0",
    "coal CCS": "#000000", "coal": "#000000",
    "refined liquids CCS": "#FFB785", "refined liquids": "#EF0C0C",
    "DAC": "#FFFFFF",
    # feedstock variants
    "coal (feedstock)": "grey",
    "refined liquids (feedstock)": "pink",
}
FUEL_PATTERNS = {"coal CCS": "/", "gas CCS": "/", "refined liquids CCS": "/", "biomass CCS": "/", "DAC": "\\"}

SECTOR_COLORS = {
    "DAC": "#FFFFFF", "Iron and Steel": "#000000", "Chemical": "#EF0C0C", "Cement": "grey",
    "Construction": "#FED30B", "Food": "#0057B5", "Paper": "green",
    "Mining": "orange", "Agriculture": "rgb(166,86,40)", "Other Industry": "#0186E0",
}
SECTOR_PATTERNS = {"DAC": "\\"}


In [22]:
# =========================================================
# DB helpers
# =========================================================
def connect_db():
    return gcamreader.LocalDBConn(DB_REL_PATH, DB_FILE)

def run_query(conn, query_idx):
    queries = gcamreader.parse_batch_query(os.fspath(QUERY_FILE))
    q = queries[query_idx]
    df = conn.runQuery(q, scenarios=SCENARIOS, regions=[REGION])
    df["scenario"] = df["scenario"].str.split(",").str[0]
    return df, q.title

In [23]:
# =========================================================
# Categorization helpers (mapping-based, no long if/elif)
# =========================================================
TECH_MAP = {
    "BLASTFUR": "BF",
    "BLASTFUR CCS": "BF-CCS",
    "BLASTFUR with hydrogen": "BF-H2",
    "Biomass-based": "BF-Biomass",
    "EAF with DRI": "DRI-EAF",
    "EAF with DRI CCS": "DRI-EAF-CCS",
    "Hydrogen-based DRI": "DRI-EAF-H2",
    "EAF with scrap": "EAF-scrap",
}

def cat_tech(series: pd.Series) -> pd.Series:
    return series.map(TECH_MAP)

SECTOR_MAP = {
    "agricultural energy use": "Agriculture",
    "ammonia": "Chemical",
    "chemical energy use": "Chemical",
    "chemical feedstocks": "Chemical",
    "cement": "Cement",
    "process heat cement": "Cement",
    "construction energy use": "Construction",
    "construction feedstocks": "Construction",
    "food processing": "Food",
    "process heat food processing": "Food",
    "iron and steel": "Iron and Steel",
    "mining energy use": "Mining",
    "paper": "Paper",
    "process heat paper": "Paper",
    "waste biomass for paper": "Paper",
    "CO2 removal": "DAC",
    "process heat dac": "DAC",
    "other industrial energy use": "Other Industry",
    "other industrial feedstocks": "Other Industry",
}

def cat_sector(series: pd.Series) -> pd.Series:
    return series.map(SECTOR_MAP).fillna("others")

def cat_fuel_ind(row) -> str:
    tech = row["technology"]
    inp  = row["input"]
    if tech in {"gas CCS", "biomass CCS", "coal CCS", "refined liquids CCS"}:
        return tech
    if inp in {"H2 wholesale dispensing", "H2 wholesale delivery", "H2 industrial"}:
        return "hydrogen"
    if inp == "elect_td_ind":
        return "electricity"
    if inp in {"refined liquids industrial"}:
        return "refined liquids"
    if inp in {"delivered biomass", "regional woodpulp for energy"}:
        return "biomass"
    if inp in {"wholesale gas"}:
        return "gas"
    if inp in {"delivered coal"}:
        return "coal"
    if tech == "electricity with solar":
        return "electricity"
    if tech == "gas with solar":
        return "gas"
    return "others"

def cat_energy_chemical(row) -> str:
    sec, inp, tech = row["sector"], row["input"], row["technology"]
    if sec == "chemical feedstocks":
        if inp == "delivered coal": return "coal (feedstock)"
        if inp == "refined liquids industrial": return "refined liquids (feedstock)"
        return "others"
    return cat_fuel_ind(row)

def cat_energy_cement(row) -> str:
    # reuse generic fuel classifier
    return cat_fuel_ind(row)

def cat_energy_other(row) -> str:
    # add "(feedstock)" suffix where applicable
    sec, base = row["sector"], cat_fuel_ind(row)
    suffix = " (feedstock)" if "feedstocks" in sec and base not in {"others"} else ""
    return f"{base}{suffix}"

In [24]:
# =========================================================
# Plot helper: two-panel stacked bar with single legend & y-title
# =========================================================
def two_panel_stack(
    df, category, years, out_png, y_max, colors, patterns=None,
    y_title=Y_TITLE, subplot_titles=("Current Policies", "Enhanced Ambition")
):
    # Aggregate
    df = df[df["Year"] >= min(years)]
    grp = df.groupby(["scenario", "Year", category], as_index=False)["value"].sum()

    # Order categories by provided palette
    order = [k for k in colors.keys() if k in grp[category].unique()]
    grp[category] = pd.Categorical(grp[category], categories=order, ordered=True)
    grp = grp.sort_values(["scenario", "Year", category])

    # Split scenarios
    left, right = grp[grp["scenario"] == SCENARIOS[0]], grp[grp["scenario"] == SCENARIOS[1]]

    fig = make_subplots(
        rows=1, cols=2, shared_yaxes=True, shared_xaxes=True,
        subplot_titles=subplot_titles
    )

    def add_side(data, col, show_legend):
        for cat in order:
            sub = data[data[category] == cat]
            if sub.empty: 
                continue
            mk = dict(color=colors.get(cat))
            if patterns and cat in patterns:
                mk["pattern"] = dict(shape=patterns[cat])
            fig.add_bar(
                name=cat, x=sub["Year"], y=sub["value"],
                marker=mk, showlegend=show_legend, row=1, col=col
            )

    # Legend only on the left
    add_side(left,  col=1, show_legend=True)
    add_side(right, col=2, show_legend=False)

    # Layout
    fig.update_layout(
        barmode="stack",
        plot_bgcolor="rgba(0,0,0,0)",
        legend=dict(traceorder="reversed", font=dict(size=16), x=1.02, y=1),
        title=dict(font=dict(size=24), x=0.5),
        width=FIG_KW["width"], height=FIG_KW["height"],
    )
    # X axes
    fig.update_xaxes(
        tickvals=years, ticktext=[str(y) for y in years], tickangle=45,
        tickfont=dict(size=14), title_font=dict(size=16)
    )
    # Left y-axis with title, right y-axis without
    fig.update_yaxes(
        title=y_title, range=[0, y_max], showgrid=True, gridcolor="lightgray",
        tickfont=dict(size=14), title_font=dict(size=16), row=1, col=1
    )
    fig.update_yaxes(
        range=[0, y_max], showgrid=True, gridcolor="lightgray",
        tickfont=dict(size=14), row=1, col=2
    )
    # Subplot titles
    fig.update_annotations(font=dict(size=18))

    pio.write_image(fig, out_png, **FIG_KW)
    return fig

In [25]:
conn = connect_db()

# ------------------------------
# A) Iron & steel production by technology
# ------------------------------
q_idx = 119
df, _ = run_query(conn, q_idx)
df["tech"] = cat_tech(df["technology"])
df = df.dropna(subset=["tech"])
dfA = (
    df[["scenario", "Year", "tech", "value"]]
    .rename(columns={"tech": "category"})
    .copy()
)
two_panel_stack(
    dfA.rename(columns={"category": "tech"}), category="tech",
    years=YEARS_2005_2050, out_png="./fig/ironsteel_production.png",
    y_max=89, colors=TECH_COLORS, patterns=TECH_PATTERNS, y_title="Mt"
)

Database scenarios: Current-Policies-Med, Enhanced-Ambition-Med, Current-Policies-High, Current-Policies-Low, Enhanced-Ambition-High, Enhanced-Ambition-Low


In [26]:
# ------------------------------
# B) Industry final energy by fuel (all industry)
# ------------------------------
q_idx = 100
df, _ = run_query(conn, q_idx)
df["fuel"] = df.apply(cat_fuel_ind, axis=1)
dfB = df[df["Year"] >= 2005].copy()
dfB["value"] *= EJ_TO_MTOE
two_panel_stack(
    dfB, category="fuel", years=list(range(2005, 2036, 5)),
    out_png="./fig/industry_energy_type.png", y_max=150,
    colors=FUEL_COLORS, patterns=FUEL_PATTERNS
)

In [27]:
# ------------------------------
# C) Industry energy by sector
# ------------------------------
dfC, _ = run_query(conn, q_idx)  # reuse q_idx=100
dfC["ind"] = cat_sector(dfC["sector"])
dfC = dfC[dfC["Year"] >= 2015].copy()
dfC["value"] *= EJ_TO_MTOE
two_panel_stack(
    dfC, category="ind", years=YEARS_2015_2050,
    out_png="./fig/industry_energy_sector.png", y_max=150,
    colors=SECTOR_COLORS, patterns=SECTOR_PATTERNS
)

In [28]:
# ------------------------------
# D) Iron & steel energy by fuel
# ------------------------------
dfD, _ = run_query(conn, q_idx)  # q_idx=100
dfD["fuel"] = dfD.apply(cat_fuel_ind, axis=1)
dfD["ind"]  = cat_sector(dfD["sector"])
dfD = dfD[(dfD["Year"] >= 2015) & (dfD["ind"] == "Iron and Steel")].copy()
dfD["value"] *= EJ_TO_MTOE
two_panel_stack(
    dfD, category="fuel", years=YEARS_2015_2050,
    out_png="./fig/ironsteel_energy_type.png", y_max=33,
    colors=FUEL_COLORS, patterns=FUEL_PATTERNS
)

In [29]:
# ------------------------------
# E) Chemical energy by fuel (incl. feedstock)
# ------------------------------
dfE, _ = run_query(conn, q_idx)  # q_idx=100
dfE["ind"]    = cat_sector(dfE["sector"])
dfE["energy"] = dfE.apply(cat_energy_chemical, axis=1)
dfE = dfE[(dfE["Year"] >= 2015) & (dfE["ind"] == "Chemical")].copy()
dfE["value"] *= EJ_TO_MTOE
two_panel_stack(
    dfE, category="energy", years=YEARS_2015_2050,
    out_png="./fig/chem.png", y_max=65,
    colors=FUEL_COLORS, patterns=FUEL_PATTERNS
)


In [30]:
# ------------------------------
# F) Cement energy by fuel
# ------------------------------
dfF, _ = run_query(conn, q_idx)  # q_idx=100
dfF["ind"]    = cat_sector(dfF["sector"])
dfF["energy"] = dfF.apply(cat_energy_cement, axis=1)
dfF = dfF[(dfF["Year"] >= 2015) & (dfF["ind"] == "Cement")].copy()
dfF["value"] *= EJ_TO_MTOE
two_panel_stack(
    dfF, category="energy", years=YEARS_2015_2050,
    out_png="./fig/cement.png", y_max=5.3,
    colors=FUEL_COLORS, patterns=FUEL_PATTERNS
)

In [31]:
# ------------------------------
# G) Other industry energy by fuel (excl. I&S, Chemical, Cement)
# ------------------------------
dfG, _ = run_query(conn, q_idx)  # q_idx=100
dfG["ind"]    = cat_sector(dfG["sector"])
dfG["energy"] = dfG.apply(cat_energy_other, axis=1)
mask_other = ~dfG["ind"].isin(["Iron and Steel", "Chemical", "Cement"])
dfG = dfG[(dfG["Year"] >= 2015) & mask_other].copy()
dfG["value"] *= EJ_TO_MTOE
two_panel_stack(
    dfG, category="energy", years=YEARS_2015_2050,
    out_png="./fig/otherInd.png", y_max=53,
    colors=FUEL_COLORS, patterns=FUEL_PATTERNS
)